In [1]:
import time
import csv
import random
import psutil
import os
import json
import requests
from datetime import datetime

# ================= 1. CẤU HÌNH CHUẨN (ĐÃ KHỚP VỚI UNIFIED_SERVER.PY) =================
TEST_DURATION_HOURS = 24        # thời gian chạy test (giờ)
REQUEST_INTERVAL = 60          # Nghỉ 60 giây giữa các request
LOG_FILE = "ket_qua_test_thuc_nghiem.csv"
DATA_FILE = "full_rag_test_suite_V2.json"

# Cấu hình API Server
# Lưu ý: Unified Server của bạn thường chạy port 8000
API_URL = "http://localhost:8000/chat"   
JSON_INPUT_KEY = "question"  # Tên biến đúng trong payload JSON

# ================= 2. HÀM ĐỌC DATA TỪ FILE CỦA BẠN =================
def load_test_dataset(filepath):
   
    if not os.path.exists(filepath):
        print(f"LỖI: Không tìm thấy file '{filepath}'")
        print("Hãy copy file 'full_rag_test_suite_V2.json' vào cùng thư mục!")
        return []

    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        questions = []
        # Duyệt qua danh sách các object trong file JSON của bạn
        for item in data:
            if "input_text" in item:
                questions.append(item["input_text"])
        
        if not questions:
            print("File JSON hợp lệ nhưng không tìm thấy trường 'input_text'.")
            return []

        print(f"Đã nạp thành công bộ dữ liệu: {len(questions)} câu hỏi.")
        return questions
    except Exception as e:
        print(f"❌ Lỗi đọc file JSON: {e}")
        return []

# ================= 3. HÀM GỌI API (ĐÃ SỬA PAYLOAD) =================
def call_api(query):
    # Tạo payload đúng chuẩn ChatRequest(question=...)
    payload = {JSON_INPUT_KEY: query} 
    
    try:
        # Timeout 45s vì xử lý Bus/Tourism có thể hơi lâu
        response = requests.post(API_URL, json=payload, timeout=45)
        
        if response.status_code == 200:
            return "SUCCESS"
        # Xử lý riêng lỗi 422 (Sai định dạng dữ liệu) để debug
        elif response.status_code == 422:
            print("⚠️ Lỗi 422: Sai format JSON (Kiểm tra lại tên biến 'question')")
            return "HTTP_422"
        else:
            return f"HTTP_{response.status_code}"
            
    except requests.exceptions.ConnectionError:
        return "CONNECTION_ERROR"
    except requests.exceptions.Timeout:
        return "TIMEOUT"
    except Exception as e:
        return "ERROR"

# ================= 4. ENGINE CHẠY TEST (LẶP VÔ TẬN) =================
def run_stability_test():
    # Bước 1: Load Data
    dataset = load_test_dataset(DATA_FILE)
    if not dataset: return

    print(f"╔══════════════════════════════════════════════════╗")
    print(f"║    STRESS TEST CHO UNIFIED SERVER (ACER NITRO 5) ║")
    print(f"║    Target API:    {API_URL:<30} ║")
    print(f"║    Input Key:     '{JSON_INPUT_KEY}' (Correct)          ║")
    print(f"║    Số lượng mẫu:  {len(dataset)} câu (Random Loop)      ║")
    print(f"║    Thời gian:     {TEST_DURATION_HOURS} giờ                        ║")
    print(f"╚══════════════════════════════════════════════════╝")

    # Bước 2: Tạo File Log
    file_exists = os.path.exists(LOG_FILE)
    with open(LOG_FILE, 'a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["Timestamp", "Question", "Latency(s)", "Status", "RAM(MB)", "CPU(%)"])

    start_time = time.time()
    end_time = start_time + (TEST_DURATION_HOURS * 3600)
    
    total_req = 0
    errors = 0
    
    try:
        while time.time() < end_time:
            iter_start = time.time()
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            
            # Chọn ngẫu nhiên câu hỏi từ file JSON
            query = random.choice(dataset)
            
            # Đo tài nguyên máy
            cpu_pct = psutil.cpu_percent(interval=None)
            mem = psutil.virtual_memory()
            ram_used_gb = round(mem.used / 1024 / 1024 / 1024, 2)
            
            # Gọi API
            t0 = time.time()
            status = call_api(query)
            latency = round(time.time() - t0, 3)
            
            # Ghi Log
            with open(LOG_FILE, 'a', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow([timestamp, query, latency, status, ram_used_gb, cpu_pct])
            
            # In ra màn hình console
            if status != "SUCCESS":
                errors += 1
                print(f"[{timestamp}] ❌ {status} | Latency: {latency}s")
            else:
                # Cắt ngắn câu hỏi hiển thị cho gọn
                short_q = (query[:45] + '...') if len(query) > 45 else query
                print(f"[{timestamp}] ✅ {latency}s | RAM: {ram_used_gb}GB | Q: {short_q}")
            
            total_req += 1
            
            # Ngủ giữ nhịp 60s
            elapsed = time.time() - iter_start
            time.sleep(max(0, REQUEST_INTERVAL - elapsed))

    except KeyboardInterrupt:
        print("\nĐã dừng test thủ công (Ctrl+C).")

    finally:
        run_hours = (time.time() - start_time) / 3600
        print(f"\n===== TỔNG KẾT SAU {run_hours:.2f} GIỜ =====")
        print(f"Tổng requests: {total_req}")
        print(f"Số lỗi:        {errors}")
        if total_req > 0:
            print(f"Tỷ lệ OK:      {((total_req-errors)/total_req)*100:.2f}%")
        print(f"File log:      {LOG_FILE}")

if __name__ == "__main__":
    # Cài thư viện trước khi chạy: pip install requests psutil
    run_stability_test()

Đã nạp thành công bộ dữ liệu: 200 câu hỏi.
╔══════════════════════════════════════════════════╗
║    STRESS TEST CHO UNIFIED SERVER (ACER NITRO 5) ║
║    Target API:    http://localhost:8000/chat     ║
║    Input Key:     'question' (Correct)          ║
║    Số lượng mẫu:  200 câu (Random Loop)      ║
║    Thời gian:     24 giờ                        ║
╚══════════════════════════════════════════════════╝
[2026-01-13 21:28:51] ✅ 2.086s | RAM: 15.05GB | Q: Từ Thảo Cầm Viên đi Chợ Bến Thành thì bắt xe ...
[2026-01-13 21:29:51] ✅ 8.411s | RAM: 14.86GB | Q: Tư vấn khách sạn gần Đại Nội Huế ở Đà Lạt.
[2026-01-13 21:30:51] ✅ 3.133s | RAM: 14.98GB | Q: Làm sao để đi xe buýt từ Landmark 81 đến Nhà ...
[2026-01-13 21:31:51] ✅ 6.568s | RAM: 14.97GB | Q: Top 5 địa điểm check-in đẹp nhất tại Buôn Ma ...
[2026-01-13 21:32:51] ✅ 3.93s | RAM: 13.93GB | Q: Làm sao đi từ TP.HCM ra Hội An bằng máy bay?
[2026-01-13 21:33:51] ✅ 5.521s | RAM: 13.96GB | Q: Từ Bến xe An Sương đi Khu du lịch Suối Tiên t...
[20